In [198]:
import itertools
import os

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, IntegerType,FloatType

from xgboost.spark import SparkXGBRegressor


In [199]:
os.getcwd()

dataset_path = '/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/eda_sdsc/data/sample_data/antarctica_sparse_features_sample.parquet/'

os.path.exists(dataset_path)

True

In [200]:


spark = SparkSession.builder \
    .appName("naive_model") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()


df_raw = spark.read.parquet(dataset_path)




In [201]:

dataset_row_count = df_raw.count()

column_names = df_raw.columns

print(f"number of rows in sample: {dataset_row_count:,}")
print(f"number of columns: {len(column_names)}")
print(f"columns: {column_names}")

number of rows in sample: 299,716
number of columns: 28
columns: ['y', 'x', 'exact_time', 'mascon_id', 'surface', 'bed', 'thickness', 'bed_slope', 'dist_to_grounding_line', 'clamped_depth', 'dist_to_ocean', 'ice_draft', 'delta_h', 'ice_area', 'surface_slope', 'h_surface_dynamic', 'thetao_mo', 't_star_mo', 'so_mo', 't_f_mo', 't_star_quarterly_avg', 't_star_quarterly_std', 'thetao_quarterly_avg', 'thetao_quarterly_std', 'lwe_mo', 'lwe_quarterly_avg', 'lwe_quarterly_std', 'lwe_fused']


In [202]:
df_raw.select('bed').head(50)

[Row(bed=1392.0),
 Row(bed=810.0),
 Row(bed=619.0),
 Row(bed=81.0),
 Row(bed=-705.0),
 Row(bed=210.0),
 Row(bed=-251.0),
 Row(bed=721.0),
 Row(bed=339.0),
 Row(bed=76.0),
 Row(bed=628.0),
 Row(bed=327.0),
 Row(bed=-279.0),
 Row(bed=-245.0),
 Row(bed=63.0),
 Row(bed=-172.0),
 Row(bed=-1671.0),
 Row(bed=515.0),
 Row(bed=411.0),
 Row(bed=466.0),
 Row(bed=346.0),
 Row(bed=630.0),
 Row(bed=510.0),
 Row(bed=-88.0),
 Row(bed=427.0),
 Row(bed=19.0),
 Row(bed=70.0),
 Row(bed=922.0),
 Row(bed=11.0),
 Row(bed=-212.0),
 Row(bed=250.0),
 Row(bed=225.0),
 Row(bed=192.0),
 Row(bed=-893.0),
 Row(bed=-1077.0),
 Row(bed=168.0),
 Row(bed=1603.0),
 Row(bed=1278.0),
 Row(bed=461.0),
 Row(bed=368.0),
 Row(bed=12.0),
 Row(bed=439.0),
 Row(bed=0.0),
 Row(bed=741.0),
 Row(bed=656.0),
 Row(bed=-440.0),
 Row(bed=-347.0),
 Row(bed=-293.0),
 Row(bed=-708.0),
 Row(bed=526.0)]

In [203]:
from pyspark.ml.classification import LogisticRegression

from pyspark.sql import functions as F
import math

In [204]:
feature_columns=[
    'surface', 'bed', 'thickness', 'bed_slope', 'dist_to_grounding_line', 
    'clamped_depth', 'dist_to_ocean', 'ice_draft', 'thetao_mo', 't_star_mo', 
    'so_mo', 't_f_mo', 't_star_quarterly_avg', 't_star_quarterly_std', 'thetao_quarterly_avg', 'thetao_quarterly_std'
    ]

label_column='lwe_fused'

df_clean = df_raw.dropna(subset=feature_columns+[label_column])

train_df, test_df = df_clean.randomSplit([0.7, 0.3], seed=42)

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

In [205]:


train_df = assembler.transform(train_df).select("features", label_column)
test_df = assembler.transform(test_df).select("features", label_column)

In [206]:

# -------------------------------
# 6️⃣ Define hyperparameter grid
# -------------------------------
param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "reg_alpha": [0, 1, 2],
    "reg_lambda": [1, 3, 5],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.7, 0.8]
}

# Generate all combinations
param_combinations = list(itertools.product(
    param_grid["max_depth"],
    param_grid["learning_rate"],
    param_grid["reg_alpha"],
    param_grid["reg_lambda"],
    param_grid["subsample"],
    param_grid["colsample_bytree"]
))

# -------------------------------
# 7️⃣ Train + evaluate each combination
# -------------------------------
best_r2 = float("-inf")
best_model = None
best_params = None

for (max_depth, lr, alpha, lam, subsample, colsample) in param_combinations:
    xgb = SparkXGBRegressor(
        features_col="features",
        label_col=label_column,
        objective="reg:squarederror",
        max_iter=400,
        max_depth=max_depth,
        learning_rate=lr,
        reg_alpha=alpha,
        reg_lambda=lam,
        subsample=subsample,
        colsample_bytree=colsample
    )
    
    model = xgb.fit(train_df)
    preds = model.transform(test_df)
    
    r2 = RegressionEvaluator(labelCol=label_column, predictionCol="prediction", metricName="r2").evaluate(preds)
    rmse = RegressionEvaluator(labelCol=label_column, predictionCol="prediction", metricName="rmse").evaluate(preds)
    
    print(f"Params: max_depth={max_depth}, lr={lr}, alpha={alpha}, lambda={lam}, subsample={subsample}, colsample={colsample}")
    print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}\n")
    
    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_params = (max_depth, lr, alpha, lam, subsample, colsample)

# -------------------------------
# 8️⃣ Best model summary
# -------------------------------
print(f"Best R2: {best_r2:.4f}")
print(f"Best params: max_depth={best_params[0]}, learning_rate={best_params[1]}, reg_alpha={best_params[2]}, reg_lambda={best_params[3]}, subsample={best_params[4]}, colsample_bytree={best_params[5]}")

2026-02-27 20:07:17,683 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.7, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.7, 'max_iter': 400, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-27 20:07:19,855 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[20:07:20] Task 0 got rank 0
/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:07:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_iter" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[20:07:21] [0]	training-rmse:60.88061
[20:07:21] [1]	training-rmse:59.81038
[20:07:21] [2]	training-rmse:58.81612
[20:07:21] [3]	training-rmse:57.74209
[20:07:21] [4]	training-rmse:56

Params: max_depth=4, lr=0.05, alpha=0, lambda=1, subsample=0.7, colsample=0.7
Test RMSE: 42.2616, R2: 0.3566



2026-02-27 20:07:24,944 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[20:07:26] Task 0 got rank 0
/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:07:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_iter" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[20:07:26] [0]	training-rmse:60.88061
[20:07:26] [1]	training-rmse:59.81350
[20:07:26] [2]	training-rmse:58.81833
[20:07:26] [3]	training-rmse:57.74360
[20:07:26] [4]	training-rmse:56.77976
[20:07:26] [5]	training-rmse:55.60600
[20:07:26] [6]	training-rmse:54.70838
[20:07:26] [7]	training-rmse:53.84106
[20:07:26] [8]	training-rmse:52.96495
[20:07:26] [9]	training-rmse:52.09119
[20:07:26] [10]	training-rmse:51.04219
[20:07:26] [11]	training-rmse:50.24926
[20:07:26] [12]	training-rmse:49.65430
[20:07:26] [13]	training-rmse:48.97707
[20:07:26] [14]	training-rmse:48.54361
[20:07:26] [15]	training-rmse:47.88

Params: max_depth=4, lr=0.05, alpha=0, lambda=1, subsample=0.7, colsample=0.8
Test RMSE: 42.2812, R2: 0.3560



2026-02-27 20:07:28,631 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[20:07:29] Task 0 got rank 0
/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:07:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_iter" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[20:07:29] [0]	training-rmse:60.78522
[20:07:29] [1]	training-rmse:59.73490
[20:07:29] [2]	training-rmse:58.77142
[20:07:29] [3]	training-rmse:57.76765
[20:07:29] [4]	training-rmse:56.79122
[20:07:29] [5]	training-rmse:55.71978
[20:07:29] [6]	training-rmse:54.94510
[20:07:29] [7]	training-rmse:54.05947
[20:07:29] [8]	training-rmse:53.09354
[20:07:29] [9]	training-rmse:52.22446
[20:07:29] [10]	training-rmse:51.21007
[20:07:29] [11]	training-rmse:50.41558
[20:07:29] [12]	training-rmse:49.86870
[20:07:29] [13]	training-rmse:49.26094
[20:07:29] [14]	training-rmse:48.71038
[20:07:29] [15]	training-rmse:48.00

...

Params: max_depth=8, lr=0.1, alpha=2, lambda=5, subsample=0.7, colsample=0.8
Test RMSE: 43.3894, R2: 0.3218



2026-02-27 20:20:07,525 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[20:20:08] Task 0 got rank 0
/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:20:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_iter" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[20:20:08] [0]	training-rmse:59.35401
[20:20:08] [1]	training-rmse:57.31486
[20:20:08] [2]	training-rmse:55.43073
[20:20:08] [3]	training-rmse:53.49869
[20:20:08] [4]	training-rmse:51.65191
[20:20:08] [5]	training-rmse:49.52131
[20:20:08] [6]	training-rmse:48.20420
[20:20:08] [7]	training-rmse:46.55457
[20:20:08] [8]	training-rmse:44.82816
[20:20:08] [9]	training-rmse:43.37555
[20:20:08] [10]	training-rmse:41.80162
[20:20:08] [11]	training-rmse:40.20756
[20:20:08] [12]	training-rmse:39.36406
[20:20:08] [13]	training-rmse:38.31228
[20:20:08] [14]	training-rmse:37.33489
[20:20:08] [15]	training-rmse:36.26

Params: max_depth=8, lr=0.1, alpha=2, lambda=5, subsample=0.8, colsample=0.7
Test RMSE: 43.3920, R2: 0.3217



2026-02-27 20:20:11,395 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[20:20:12] Task 0 got rank 0
/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:20:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_iter" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[20:20:12] [0]	training-rmse:59.28683
[20:20:12] [1]	training-rmse:57.26523
[20:20:12] [2]	training-rmse:55.37243
[20:20:12] [3]	training-rmse:53.41115
[20:20:12] [4]	training-rmse:51.58166
[20:20:12] [5]	training-rmse:49.47748
[20:20:12] [6]	training-rmse:48.07857
[20:20:12] [7]	training-rmse:46.43937
[20:20:12] [8]	training-rmse:44.74300
[20:20:12] [9]	training-rmse:43.14433
[20:20:12] [10]	training-rmse:41.41233
[20:20:12] [11]	training-rmse:39.78116
[20:20:12] [12]	training-rmse:38.92077
[20:20:12] [13]	training-rmse:37.99455
[20:20:12] [14]	training-rmse:36.99164
[20:20:12] [15]	training-rmse:35.99

Params: max_depth=8, lr=0.1, alpha=2, lambda=5, subsample=0.8, colsample=0.8
Test RMSE: 42.3281, R2: 0.3546

Best R2: 0.3781
Best params: max_depth=6, learning_rate=0.05, reg_alpha=2, reg_lambda=5, subsample=0.8, colsample_bytree=0.8


In [207]:
xgb = SparkXGBRegressor(
    features_col="features",
    label_col=label_column,
    objective="reg:squarederror",
    max_iter=400,          # boosting rounds
    max_depth=12,           # tree depth
    learning_rate=0.02,    # step size shrinkage
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=3,
    reg_lambda=7
)

In [208]:
model = xgb.fit(train_df)

2026-02-27 20:20:14,269 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.02, 'max_depth': 12, 'min_child_weight': 3, 'reg_alpha': 3, 'reg_lambda': 7, 'subsample': 0.8, 'max_iter': 400, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-02-27 20:20:15,392 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[20:20:16] Task 0 got rank 0
/home/hansp/Documents/DSC232R_bigdata/dsc232_group_project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:20:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_iter" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[20:20:16] [0]	training-rmse:61.43709
[20:20:16] [1]	training-rmse:61.03235
[20:20:16] [2]	training-rmse:60.64210
[20:20:16] [3]	training-rmse:60.19293
[20:20:

In [209]:
predictions = model.transform(test_df) 
predictions.select("features", label_column, "prediction").show(10)

2026-02-27 20:20:18,732 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


+--------------------+--------------------+--------------------+
|            features|           lwe_fused|          prediction|
+--------------------+--------------------+--------------------+
|[47.0,-736.0,300....|  1.3483222643813952|   3.207475423812866|
|[46.0,-616.0,293....| 0.27627969339955294| -2.0669643878936768|
|[44.0,-675.0,254....|  0.3857023820092592|  1.8050204515457153|
|[46.0,-641.0,285....|0.058283751264171645|  1.7220898866653442|
|[46.0,-619.0,296....| 0.22472676302769584| -1.6921629905700684|
|[44.0,-679.0,264....| 0.14921185308771232|  0.3857293426990509|
|[45.0,-669.0,277....|  0.8852675707617143| -1.5554603338241577|
|[47.0,-622.0,297....|   0.143395837531424| -2.0672919750213623|
|[47.0,-649.0,291....| 0.07350456147501384| -2.8016343116760254|
|[48.0,-712.0,295....|  3.7346658818347493|-0.16996559500694275|
+--------------------+--------------------+--------------------+
only showing top 10 rows


In [210]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol=label_column,
    predictionCol="prediction",
    metricName="rmse"  # options: "rmse", "mae", "r2"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse:.4f}")

r2 = RegressionEvaluator(
    labelCol=label_column,
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions)
print(f"R2: {r2:.4f}")

2026-02-27 20:20:19,002 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


RMSE: 42.0361
R2: 0.3635


2026-02-27 20:20:20,105 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


In [211]:
df_raw.select("time").distinct().count()

{"ts": "2026-02-27 20:20:20.249", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `time` cannot be resolved. Did you mean one of the following? [`x`, `y`, `bed`, `so_mo`, `t_f_mo`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor72.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o6127.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `time` cannot be resolved. Did you mean one of the following? [`x`, `y`, `bed`, `so_mo`, `t_f_mo`]. SQLSTATE: 42703;\n'Project ['time]\n+- Relation [y#1835,x#1836,exact_time#1837,mascon_id#1838,surface#1839,bed#1840,thickness#1841,bed_slope#1842,dist_to_grounding_line#1843,clamped_depth#1844

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `time` cannot be resolved. Did you mean one of the following? [`x`, `y`, `bed`, `so_mo`, `t_f_mo`]. SQLSTATE: 42703;
'Project ['time]
+- Relation [y#1835,x#1836,exact_time#1837,mascon_id#1838,surface#1839,bed#1840,thickness#1841,bed_slope#1842,dist_to_grounding_line#1843,clamped_depth#1844,dist_to_ocean#1845,ice_draft#1846,delta_h#1847,ice_area#1848,surface_slope#1849,h_surface_dynamic#1850,thetao_mo#1851,t_star_mo#1852,so_mo#1853,t_f_mo#1854,t_star_quarterly_avg#1855,t_star_quarterly_std#1856,thetao_quarterly_avg#1857,thetao_quarterly_std#1858,lwe_mo#1859,... 3 more fields] parquet


In [ ]:
df_filtered = df_raw.dropna(subset=["ice_thickness"])

In [ ]:
#doing time calcs on full df so that I can sample properly

In [ ]:
df_filtered = df_filtered.withColumn(
    "time",
    # first convert nanoseconds → seconds (long)
    (F.col("time") / 1_000_000_000).cast("long")
).withColumn(
    # convert seconds since epoch → timestamp
    "time",
    F.from_unixtime(F.col("time")).cast("date")
)

In [ ]:
df_filtered.select('time').head(5)

In [ ]:
df_filtered = df_filtered.withColumn(
    "day_of_year",
    F.dayofyear(F.col("time"))
)

In [ ]:
# Get fraction per day-of-year (e.g., 20% for each day)
# fractions = df_filtered.select("day_of_year").distinct().withColumn("fraction", F.lit(0.2)) \
#     .rdd.collectAsMap()


# df_sampled = df_filtered.sampleBy("day_of_year", fractions, seed=42)

#faster than doing the sort of above method
from pyspark.sql.window import Window

w = Window.partitionBy("x", "y").orderBy(F.rand())

df_sampled = df_filtered.withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") <= 5)  # take up to 5 rows per (x, y)


dataset_row_count = df_sampled.count()

column_names = df_sampled.columns

print(f"number of rows in sample: {dataset_row_count:,}")
print(f"number of columns: {len(column_names)}")
print(f"columns: {column_names}")

In [ ]:
# df.select('day_of_year').head(5)
# df.select('time').head(5)

df_sampled = df_sampled.withColumn(
    "sin_time",
    F.sin(2 * F.lit(math.pi) * F.col("day_of_year") / F.lit(365))
)

df_sampled = df_sampled.withColumn(
    "cos_time",
    F.cos(2 * F.lit(math.pi) * F.col("day_of_year") / F.lit(365))
)

In [ ]:
df_sampled.select("day_of_year", "sin_time", "cos_time").show(10)

In [ ]:
print(f"columns: {column_names}")

In [ ]:
from pyspark.ml.feature import VectorAssembler

featurecolumns=[
    'surface_topography',
    'bed_topography',
    
]

assembler = VectorAssembler(
    inputCols=["x", "y"],
    outputCol="features"
)

df_model = assembler.transform(df_sampled)



In [ ]:
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# from pyspark.ml.regression import LinearRegression

# lr = LinearRegression(
#     featuresCol="features",
#     labelCol="ice_thickness",
#     regParam=0.1
# )

# model = lr.fit(train_df)

In [ ]:
print("Intercept:", model.intercept)
print("Coefficients:", model.coefficients)

In [ ]:
predictions = model.transform(test_df)

In [ ]:
predictions.select(
    "x",
    "y",
    "ice_thickness",
    "prediction"
).show()